[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cafzal/frontier/blob/main/examples/exact_solver_comparison/exact_solver_comparison.ipynb)

# Exact solvers as Frontier backends — coverage · optimality · speed

Frontier's NSGA-II is an **approximate** multi-objective explorer. Drop **any exact solver** in as the
per-scalarization inner solve and every returned point becomes optimal. The exact backend is **pluggable** —
this notebook demonstrates the pattern with **two interchangeable solvers, one CPU and one GPU (HiGHS and
NVIDIA cuOpt)** — and measures **coverage · optimality · speed**: what an exact solver buys over the heuristic,
and how the two backends stack up head-to-head.

**5 configurations**
- **NSGA-alone** — the heuristic baseline (no exact solver).
- **solver-alone** — one exact solve = **one plan** (the same point on either backend).
- **NSGA + HiGHS** / **NSGA + cuOpt** — the same pairing on each backend (NSGA evolves scalarizations; the solver makes each exact).
- *(reference)* **exact frontier** — the nondominated union of every exact solve = the 100% ceiling.

**3 value vectors**
- **Coverage** — share of the exact frontier's hypervolume a config recovers (breadth).
- **Optimality** — % of a config's points that are actually Pareto-optimal vs the exact frontier (depth).
- **Speed** — wall-time per solve vs problem size.

**Two takeaways**
- **The pattern is solver-agnostic.** Both exact solvers, in the loop, make the frontier optimal —
  `NSGA+cuOpt ≈ NSGA+HiGHS`, both beating NSGA-alone (optimality at every scale, coverage at scale). The
  *choice* of exact solver doesn't change the frontier; the **exact-solver-in-the-loop pattern** is the value.
- **The solvers differ only on speed.** HiGHS (CPU) owns small/medium scale (negligible overhead); cuOpt (GPU)
  pulls ahead only at the top end, where the CPU solve climbs into the time cap and walls out — read it off Panel 2.

Synthetic data; every exact point is **optimal to a 0.1% gap** (not "certified"). Pick a **GPU runtime**
(*Runtime → Change runtime type → GPU*) and *Run all*. Panel 1's `NSGA+cuOpt` cell is the slow one
(~10–15 min of GPU solves); Panel 2 (speed) is independent and quicker.

In [ ]:
import subprocess
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                         capture_output=True, text=True)
    print(out.stdout.strip() or "no nvidia-smi output")
except FileNotFoundError:
    print("No NVIDIA GPU detected - cuOpt cannot run. In Colab: Runtime -> Change runtime type -> GPU (T4).")

In [ ]:
# Bootstrap: clone Frontier (public) + install cuOpt (GPU) + HiGHS (CPU) + engine deps. First run ~2-3 min.
import os, subprocess, sys
REPO = "/content/frontier"
if not os.path.isdir(REPO):
    rc = subprocess.run(["git", "clone", "-q", "https://github.com/cafzal/frontier.git", REPO],
                        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"}).returncode
    if rc != 0 or not os.path.isdir(REPO):
        raise RuntimeError("Could not clone github.com/cafzal/frontier - check the connection, then re-run.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--extra-index-url", "https://pypi.nvidia.com", "cuopt-cu12"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "highspy",
                "pymoo>=0.6", "pydantic>=2.0", "scipy>=1.11", "pandas>=2.0", "matplotlib>=3.7"], check=True)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print("Frontier on path:", REPO in sys.path)

In [ ]:
import json, os, time
import numpy as np
import matplotlib.pyplot as plt
from pymoo.indicators.hv import HV
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
from engine.models import Problem
from engine.optimizer import optimize
from solvers import cuopt_backend as cb     # _build_milp_data (re-export), _solve_milp_cuopt (GPU)
from solvers import highs_backend as hb     # _solve_milp_highs (CPU exact)
# Confirms the GPU solver is live (fails if not on a cuOpt runtime):
from cuopt.linear_programming.problem import Problem as CuProblem
print("cuOpt + HiGHS import OK - both exact backends live.")

In [ ]:
_MIX = {"Growth": 30, "Digital": 20, "R&D": 20, "Maintenance": 20, "Compliance": 15, "Efficiency": 15}
_TILT = {"Growth": 1.25, "Digital": 1.15, "R&D": 1.4, "Maintenance": 0.5, "Compliance": 0.2, "Efficiency": 0.9}
_RISK = {"Growth": 55, "Digital": 45, "R&D": 78, "Maintenance": 22, "Compliance": 18, "Efficiency": 35}
_FIT = {"Growth": 70, "Digital": 72, "R&D": 60, "Maintenance": 45, "Compliance": 85, "Efficiency": 58}

def gen_capital(n, seed=11):
    """A feasible binary capital-select MILP with ~n projects (NPV/Cost/Risk/StrategicFit) under a
    budget + per-category caps + dependencies + exclusions + cardinality. Scales the committed
    120-project instance proportionally; deterministic."""
    rng = np.random.default_rng(seed)
    counts = {c: max(1, round(n * k / 120)) for c, k in _MIX.items()}
    cats = [c for c, k in counts.items() for _ in range(k)]
    n = len(cats); ids = [f"P{i+1:05d}" for i in range(n)]
    options, scores, cost_arr = [], [], []
    for pid, cat in zip(ids, cats):
        cost = float(np.round(rng.uniform(4, 45), 1))
        npv = float(np.round(max(-8, cost * _TILT[cat] * rng.uniform(0.6, 1.7) - rng.uniform(0, 10)), 1))
        risk = float(np.round(np.clip(_RISK[cat] + rng.normal(0, 12), 5, 98), 1))
        fit = float(np.round(np.clip(_FIT[cat] + rng.normal(0, 14), 10, 99), 1))
        cost_arr.append(cost); options.append({"name": pid, "description": f"{cat} initiative"})
        scores += [{"option": pid, "objective": "NPV", "value": npv}, {"option": pid, "objective": "Cost", "value": cost},
                   {"option": pid, "objective": "Risk", "value": risk}, {"option": pid, "objective": "StrategicFit", "value": fit}]
    by_cat = {c: [ids[i] for i in range(n) if cats[i] == c] for c in counts}
    budget = float(np.round(0.22 * sum(cost_arr), -1))
    forced = by_cat["Compliance"][:max(1, round(3 * n / 120))]; enablers = by_cat["Efficiency"]
    dep_src = by_cat["Growth"][:round(6 * n / 120)] + by_cat["Digital"][:round(4 * n / 120)]
    deps = [{"type": "dependency", "if_option": s, "then_option": enablers[i % len(enablers)]} for i, s in enumerate(dep_src)]
    gp, rp = by_cat["Growth"], by_cat["R&D"]; npair = min(len(gp)//2, len(rp)//2, max(1, round(5*n/120)))
    excl = ([{"type": "exclusion_pair", "option_a": gp[2*i], "option_b": gp[2*i+1]} for i in range(npair)]
            + [{"type": "exclusion_pair", "option_a": rp[2*i], "option_b": rp[2*i+1]} for i in range(npair)])
    groups = [{"type": "group_limit", "options": by_cat[c], "max": max(1, round(m*n/120))}
              for c, m in {"Growth": 8, "Digital": 6, "R&D": 6, "Maintenance": 7}.items()]
    card_lo, card_hi = max(2, round(0.15*n)), max(3, round(0.33*n))
    constraints = ([{"type": "objective_bound", "objective": "Cost", "operator": "max", "value": budget},
                    {"type": "cardinality", "min": card_lo, "max": card_hi}]
                   + [{"type": "force_include", "option": f} for f in forced] + deps + excl + groups)
    return Problem(name=f"Capital Project Selection ({n})", domain="corporate finance",
                   context=f"Choose which of {n} capital projects to fund under a ${budget:.0f}M budget.", approach="binary",
                   objectives=[{"name": "NPV", "direction": "maximize", "unit": "$M", "aggregation": "sum"},
                               {"name": "Cost", "direction": "minimize", "unit": "$M", "aggregation": "sum"},
                               {"name": "Risk", "direction": "minimize", "unit": "score", "aggregation": "sum"},
                               {"name": "StrategicFit", "direction": "maximize", "unit": "score", "aggregation": "sum"}],
                   constraints=constraints, options=options, scores=scores)

def _pts(run, objs):
    return np.array([[s.objective_values[o.name] for o in objs] for s in run.solutions])

def assemble_exact(exact_sets, dirs):
    """Exact-frontier reference = nondominated union of all EXACT-backed point sets (min-space)."""
    allpts = np.vstack([np.asarray(p, float) * dirs for p in exact_sets if len(p)])
    keep = NonDominatedSorting().do(allpts, only_non_dominated_front=True)
    return allpts[keep]

def coverage_share(pts, exact_ref_min, lo, span, ref):
    """HV(config) / HV(exact frontier) — breadth of the tradeoff surface spanned."""
    if not len(pts):
        return 0.0
    m = (np.asarray(pts, float) - lo) / span
    return float(HV(ref_point=ref)(m) / HV(ref_point=ref)((exact_ref_min - lo) / span))

def optimality_pct(pts_min, exact_ref_min, eps=1e-9):
    """% of a config's points NOT dominated by the exact frontier — per-point exactness (depth)."""
    if not len(pts_min):
        return 0.0
    return sum(not np.any(np.all(exact_ref_min <= p + eps, axis=1) & np.any(exact_ref_min < p - eps, axis=1))
               for p in pts_min) / len(pts_min)

In [ ]:
def run_configs(n, n_weights=300, seed=42):
    """Run all 5 configs on a size-n capital MILP and return points + the exact reference.
    NSGA+cuOpt is the slow rung (GPU EA loop); the sweep + solver-alone use HiGHS (same optima, far faster)."""
    problem = gen_capital(n); objs = problem.objectives
    nb, names, S, dirs, mc = cb._build_milp_data(problem)
    cmin, cmax = S.min(0), S.max(0); Snorm = (S - cmin) / np.where(cmax > cmin, cmax - cmin, 1.0)

    t = time.perf_counter(); nsga = _pts(optimize(problem, seed=seed), objs); t_nsga = time.perf_counter() - t
    t = time.perf_counter(); pair_h = _pts(optimize(problem, seed=seed, solver="highs"), objs); t_h = time.perf_counter() - t
    t = time.perf_counter(); pair_c = _pts(optimize(problem, seed=seed, solver="cuopt"), objs); t_c = time.perf_counter() - t

    solo_sel, _ = hb._solve_milp_highs(dirs[0] * S[:, 0], [], mc, nb)       # one exact plan (same for either solver)
    solo = np.array([[float(S[:, j] @ solo_sel) for j in range(len(objs))]])
    rng = np.random.default_rng(0)
    W = list(np.eye(len(objs))) + [w / w.sum() for w in rng.random((n_weights, len(objs)))]
    seen, sweep = set(), []
    for w in W:                                                            # weighted-sum exact ceiling (HiGHS, fast)
        sel, ok = hb._solve_milp_highs(Snorm @ (np.asarray(w) * dirs), [], mc, nb)
        if not ok:
            continue
        key = tuple(round(float(S[:, j] @ sel), 3) for j in range(len(objs)))
        if key not in seen:
            seen.add(key); sweep.append(list(key))
    sweep = np.array(sweep)

    exact_ref = assemble_exact([sweep, pair_c, pair_h, solo], dirs)
    return dict(objs=objs, dirs=dirs, nsga=nsga, pair_h=pair_h, pair_c=pair_c, solo=solo, sweep=sweep,
                exact_ref=exact_ref, times={"NSGA": t_nsga, "NSGA+HiGHS": t_h, "NSGA+cuOpt": t_c})

## Panel 1 — coverage + optimality (capital projects, n ≈ 120)

Coverage and optimality are measured against the **exact frontier** = the nondominated *union of every
exact solve* (the sweep + both pairings) = 100%. The pairing rungs (`NSGA+cuOpt`, `NSGA+HiGHS`) should land
**above NSGA-alone** on coverage and at **100% optimality** (every point exact) where NSGA-alone falls
short — and be **interchangeable** (the solver-agnostic pattern). *(At ≤30 vars this washes — NSGA-alone can
even cover more, since the EA is solve-budget-starved; the win is a **scale** effect, shown here at 120.
Note the weighted-sum sweep alone isn't the whole frontier — it misses non-supported points the ε-constraint
EA finds, which is why the ceiling is the union.)*

> The `NSGA+cuOpt` rung issues ~750 GPU MILP solves — **~10–15 min**. The HiGHS rung (~2–3 min) and the
> sweep/solo (seconds) are the CPU companions in the same session.

In [ ]:
R = run_configs(120)
objs, dirs, ref_min = R["objs"], R["dirs"], R["exact_ref"]
order = ["NSGA alone", "NSGA + HiGHS", "NSGA + cuOpt", "solver alone", "exact sweep"]
sets = {"NSGA alone": R["nsga"], "NSGA + HiGHS": R["pair_h"], "NSGA + cuOpt": R["pair_c"],
        "solver alone": R["solo"], "exact sweep": R["sweep"]}
allmin = np.vstack([np.asarray(p, float) * dirs for p in sets.values() if len(p)] + [ref_min])
lo, hi = allmin.min(0), allmin.max(0); span = np.where(hi > lo, hi - lo, 1.0); rp = np.full(len(objs), 1.1)
cov = {k: coverage_share(np.asarray(v, float) * dirs, ref_min, lo, span, rp) for k, v in sets.items()}
opt = {k: optimality_pct(np.asarray(v, float) * dirs, ref_min) for k, v in sets.items()}

print(f"  {'config':<14}{'#pts':>6}{'coverage':>10}{'optimality':>12}")
for k in order:
    print(f"  {k:<14}{len(sets[k]):>6}{cov[k]:>9.0%}{opt[k]:>11.0%}")

xi, yi = 0, 1   # NPV vs Cost
fig, ax = plt.subplots(1, 2, figsize=(13.5, 5.4))
sty = {"NSGA alone": ("darkorange", 46, 0.5), "NSGA + HiGHS": ("seagreen", 34, 0.8),
       "NSGA + cuOpt": ("royalblue", 34, 0.8), "exact sweep": ("navy", 14, 0.85)}
for k, (c, s, a) in sty.items():
    P = sets[k]
    if len(P):
        ax[0].scatter(P[:, xi], P[:, yi], s=s, color=c, alpha=a, label=f"{k} ({len(P)}) — {cov[k]:.0%}")
ax[0].scatter(sets["solver alone"][:, xi], sets["solver alone"][:, yi], s=150, marker="*",
              color="crimson", zorder=6, label="solver alone (1 plan)")
ax[0].set_xlabel(objs[xi].name); ax[0].set_ylabel(objs[yi].name)
ax[0].set_title("Coverage — one plan vs the frontier"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

bx = ["NSGA alone", "NSGA + HiGHS", "NSGA + cuOpt"]; x = np.arange(len(bx)); w = 0.38
ax[1].bar(x - w/2, [cov[k] for k in bx], w, color="steelblue", label="coverage")
ax[1].bar(x + w/2, [opt[k] for k in bx], w, color="indianred", label="optimality")
ax[1].set_xticks(x); ax[1].set_xticklabels([k.replace(" ", "\n") for k in bx], fontsize=9)
ax[1].set_ylim(0, 1.08); ax[1].axhline(1.0, color="gray", lw=0.8, ls="--")
ax[1].set_title("Coverage & optimality vs the exact frontier (dashed = 100%)"); ax[1].legend(); ax[1].grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

## Panel 2 — speed: does the GPU push past where the CPU walls out?

One **bounded** (0.1%-gap) single-objective MILP solve vs problem size — cuOpt (GPU) and HiGHS (CPU),
same problem, same gap and 30 s cap. This is the **per-solve** question (no EA loop): HiGHS owns the small
end (its overhead is ~ms), so cuOpt's fixed GPU overhead loses there; the open question is the **top end**,
where HiGHS's solve time climbs into the 30 s cap and eventually returns nothing. `cb_build` is cuOpt's
Python problem-build time (reported apart from the solve — it's API overhead, not the solver).

In [ ]:
cb._MILP_REL_GAP = 1e-4; cb._MILP_TIME_LIMIT = 30.0   # parity with the HiGHS backend (gap + cap)
scales = (1000, 2000, 5000, 10000, 20000, 35000, 50000)
H, C, B, NS = [], [], [], []
print(f"  {'n':>7}{'HiGHS':>10}{'cuOpt(build+solve)':>22}")
for n in scales:
    problem = gen_capital(n); nb, names, S, dirs, mc = cb._build_milp_data(problem)
    cmin, cmax = S.min(0), S.max(0); Snorm = (S - cmin) / np.where(cmax > cmin, cmax - cmin, 1.0)
    coef = Snorm @ (np.ones(len(problem.objectives)) / len(problem.objectives) * dirs)
    t = time.perf_counter(); _, okh = hb._solve_milp_highs(coef, [], mc, nb); th = time.perf_counter() - t
    t = time.perf_counter(); _, okc = cb._solve_milp_cuopt(coef, [], mc, nb); tc = time.perf_counter() - t
    H.append(th if okh else np.nan); C.append(tc); NS.append(nb)
    print(f"  {nb:>7}{(f'{th:.2f}s' if okh else 'FAIL'):>10}{f'{tc:.2f}s':>22}")

fig, ax = plt.subplots(figsize=(8, 5.4))
ax.plot(NS, H, "o-", color="darkorange", label="HiGHS (CPU)")
ax.plot(NS, C, "s-", color="royalblue", label="cuOpt (GPU)")
for n, th in zip(NS, H):
    if np.isnan(th):
        ax.scatter([n], [30], marker="x", s=90, color="red", zorder=5, label="HiGHS no solution in 30s")
ax.axhline(30, color="gray", ls="--", lw=0.8)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("problem size n (binary vars)")
ax.set_ylabel("one bounded solve (s)"); ax.set_title("Per-solve speed vs scale — cuOpt GPU vs HiGHS CPU")
h, l = ax.get_legend_handles_labels(); ax.legend(dict(zip(l, h)).values(), dict(zip(l, h)).keys())
ax.grid(alpha=0.3, which="both"); plt.tight_layout(); plt.show()

## What the comparison shows

- **Either backend makes Frontier's frontier optimal.** `NSGA+HiGHS` and `NSGA+cuOpt` both beat NSGA-alone —
  on **optimality at every scale** (100% vs a heuristic that returns sub-optimal points) and on **coverage at
  scale** — and they're **interchangeable** on both. So the value is the **exact-solver-in-the-loop pattern**,
  not the specific solver (and not GPU-specific). (Honest boundary: a direct ε-sweep is the *more efficient*
  exact method on a linear MILP; the EA pairing earns its keep on non-convex / black-box fronts; and at small
  scale it washes — NSGA-alone can cover more.)
- **The two backends differ only on speed.** HiGHS (CPU, `pip install highspy`) wins small/medium scale; the
  GPU backend matters only at the largest sizes where the CPU walls out — a scale beyond a realistic Frontier
  problem (the many-solve frontier is slow on both there).

**Net:** an exact inner solve turns Frontier's *approximate* explorer into an *optimal* frontier — with **any**
exact backend. HiGHS is the no-GPU default everywhere; a GPU solver is the speed lever at the extreme end.

*Synthetic data; every exact point optimal to a 0.1% gap; duals (a separate artifact) exist for continuous
QP only, never a MILP.*